In [65]:
import os
import sys 
os.chdir("/workspaces/dev/app")
sys.path.append("/workspaces/dev/app")

In [66]:
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from dotenv import load_dotenv

In [67]:
load_dotenv()

True

In [68]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

In [69]:
genai.configure(api_key=GOOGLE_API_KEY)

In [70]:
model = genai.GenerativeModel(
  'models/gemini-2.0-flash-lite',
  generation_config=genai.types.GenerationConfig(
    temperature=0.0,
    max_output_tokens=1024,
  )
)

In [71]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [72]:
import re

def extract_tagged_output(text: str):
    context_match = re.search(r"<context>(.*?)</context>", text, re.DOTALL)
    agenda_match = re.search(r"<agenda>(.*?)</agenda>", text, re.DOTALL)

    context = context_match.group(1).strip() if context_match else None
    agenda_raw = agenda_match.group(1).strip() if agenda_match else ""
    agenda = [int(a) for a in agenda_raw.split(",")] if agenda_raw else []

    return context, agenda

In [73]:
def print_tagged_output(cnt_msg:str, response: str, agenda:list[str]):
  print("현재 메시지" + "-" * 40)
  print(cnt_msg)
  print("응답" + "-" * 10)
  print(response)

  context, agenda_ = extract_tagged_output(response)
  if context:
    print("Context:" + "-" * 10)
    print(context)
  else:
    print("No context found.")
  
  if agenda_:
    print("\nAgenda:" + "-" * 10)
    print(agenda_)
    agenda = [a for i, a in enumerate(agenda) if i not in agenda_]
  else:
    print("No agenda found.")
  return agenda

In [79]:
def run_chat(conversation:list[str], prompt:str, last_prompt:str, agenda:list[str]):
  chat = model.start_chat(history=[])

  cnt_msg = ""
  for message in conversation:
    cnt_msg += message + "\n"
    if len(cnt_msg) > 500:
      agenda_text = ""
      for i, a in enumerate(agenda):
        agenda_text += f"{i}: {a}\n"
      cnt_msg = prompt.format(agenda_text, cnt_msg)
      response = chat.send_message(cnt_msg)
      agenda = print_tagged_output(cnt_msg, response.text, agenda)
      cnt_msg = ""

  if cnt_msg:
      agenda_text = ""
      for i, a in enumerate(agenda):
        agenda_text += f"{i}: {a}\n"
      cnt_msg = prompt.format(agenda_text, cnt_msg)
      response = chat.send_message(cnt_msg)
      print_tagged_output(cnt_msg, response.text, agenda)

  response = chat.send_message(last_prompt)
  print(response.text)
  print("-"*40)

In [88]:
prompt = (
"""
너는 회의 내용을 정리해주는 뉴스 아나운서야.

[요약 지침]
- 이번 발화 내용만 읽고, 핵심만 한 문장으로 요약해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성해.
- 마크다운, 리스트, 기호 없이 일반 텍스트로 작성해.
- 반드시 다음 형식으로 출력해:  
  <context>요약 내용</context>

[아젠다 판별 지침]
- 아래 아젠다 항목 중, 이번 발화 내용에 의해 **명확히 완료된 항목**이 있다면 번호를 출력해.
- “완료”란, 실제로 실행되었거나 결정되었다는 표현이 명확히 포함된 경우만 의미함.
- 단순히 언급만 되었거나 논의 중인 것은 완료로 간주하지 마.
- 관련된 아젠다가 없다면 빈 태그로 남겨둬.
- 반드시 다음 형식으로 출력해:  
  <agenda>1, 3</agenda>

[아젠다 항목]
{0}

[회의 내용]
{1}
"""
)

last_prompt = (
"""
너는 전체 회의 내용을 정리해주는 뉴스 아나운서야.

[요약 지침]
- 지금까지 나눈 회의 대화를 읽고, 핵심만 뽑아 요약해.
- 전체 내용을 100문장 이내로 정리하되, 주제 흐름에 따라 자연스럽게 구성해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성해.
- 마크다운, 리스트, 기호 없이 일반 텍스트로 작성해.
- 반드시 다음 형식으로 출력해:
  <context>요약 내용</context>
"""
)


In [89]:
text = None
with open("/workspaces/dev/.data/meeting_sample.txt", "r") as f:
  text = f.readlines()

In [90]:
agenda = [
  "회의 목적 설명",
  "프로젝트 진행 상황 공유",
  "이슈 및 장애물 논의",
  "다음 회의 일정 조율"
]


In [91]:
run_chat(
  conversation=text,
  prompt=prompt,
  last_prompt=last_prompt,
  agenda=agenda
)

현재 메시지----------------------------------------

너는 회의 내용을 정리해주는 뉴스 아나운서야.

[요약 지침]
- 이번 발화 내용만 읽고, 핵심만 한 문장으로 요약해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성해.
- 마크다운, 리스트, 기호 없이 일반 텍스트로 작성해.
- 반드시 다음 형식으로 출력해:  
  <context>요약 내용</context>

[아젠다 판별 지침]
- 아래 아젠다 항목 중, 이번 발화 내용에 의해 **명확히 완료된 항목**이 있다면 번호를 출력해.
- “완료”란, 실제로 실행되었거나 결정되었다는 표현이 명확히 포함된 경우만 의미함.
- 단순히 언급만 되었거나 논의 중인 것은 완료로 간주하지 마.
- 관련된 아젠다가 없다면 빈 태그로 남겨둬.
- 반드시 다음 형식으로 출력해:  
  <agenda>1, 3</agenda>

[아젠다 항목]
0: 회의 목적 설명
1: 프로젝트 진행 상황 공유
2: 이슈 및 장애물 논의
3: 다음 회의 일정 조율


[회의 내용]
사람1: 프론트엔드 쪽에서는 메인 화면 UI를 전면 개편하려고 합니다.

사람4: 앱 이름에 대한 아이디어는 따로 문서로 정리해서 공유드릴게요.

사람2: 에러 로그가 너무 많이 쌓이고 있어서 모니터링 기준을 조정할 필요가 있습니다.

사람4: 다국어 지원은 다음 분기 목표로 미뤄두는 게 좋겠습니다.

사람3: 이번 릴리즈는 4월 말까지 마무리하는 걸 목표로 하고 있습니다.

사람2: 검색 결과 정렬 로직을 개선하는 작업은 제가 맡겠습니다.

사람2: 프론트엔드 쪽에서는 메인 화면 UI를 전면 개편하려고 합니다.

사람4: 고객 피드백에 따르면 검색 기능 정확도가 낮다고 합니다.

사람3: 앱 초기 로딩 속도가 느리다는 피드백이 계속 들어오고 있어요.

사람3: 프로토타입 테스트 결과, 하단 네비게이션 바 구성이 헷갈린다는 의견이 있었어요.

사람2: Slack 알림 조건을 세분화해서 꼭 필요한 이슈만 알리도록 하겠습니다.

사람3